## Test EuroSAT-MS DataLoader

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

# Resolve project root robustly whether notebook is opened from project root or notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_ROOT = PROJECT_ROOT / "data" / "EuroSAT_MS"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("DATA_ROOT exists:", DATA_ROOT.exists())

from src.datasets.eurosat import (
    EuroSATMSDataset,
    build_eurosat_dataloaders,
    describe_split_sizes,
    extract_rgb_bands,
    get_default_text_queries,
    SENTINEL2_BANDS,
)

## Build dataset + dataloaders
split `80/10/10` using indices

In [ ]:
bundle = build_eurosat_dataloaders(
    root=DATA_ROOT,
    batch_size=16,
    num_workers=0,    
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    seed=42,
    normalize=True,
)

dataset = bundle["dataset"]
indices = bundle["indices"]
loaders = bundle["loaders"]

print("Dataset size:", len(dataset))
print("Split sizes:", describe_split_sizes(indices))
print("Class distribution:", dataset.get_class_distribution())
print("Band names:", SENTINEL2_BANDS)
print("Text prompts:", dataset.get_text_prompts())

## Inspect 1 sample
return sample:
- 13-channel image
- label
- short descriptive text

In [ ]:
sample = dataset[0]

print("Keys:", sample.keys())
print("Image shape:", sample["image"].shape)
print("Label:", sample["label"])
print("Label name:", sample["label_name"])
print("Text:", sample["text"])
print("Path:", sample["path"])
print("Image dtype:", sample["image"].dtype)
print("Image min/max:", sample["image"].min().item(), sample["image"].max().item())

## Visualize 1 sample
Display:
- RGB composite from B04/B03/B02
- All 13 bands for checking band order and data range

In [ ]:
def stretch_for_display(rgb_tensor: torch.Tensor) -> np.ndarray:
    """
    rgb_tensor: [3, H, W], assumed roughly in [0,1]
    Return: HWC numpy image for plotting
    """
    rgb = rgb_tensor.detach().cpu().numpy().transpose(1, 2, 0)
    low = np.percentile(rgb, 2)
    high = np.percentile(rgb, 98)
    rgb = np.clip((rgb - low) / (high - low + 1e-8), 0, 1)
    return rgb

image_13 = sample["image"]
rgb = extract_rgb_bands(image_13)

fig = plt.figure(figsize=(16, 8))

# RGB composite
ax = plt.subplot(3, 5, 1)
ax.imshow(stretch_for_display(rgb))
ax.set_title(f'RGB: {sample["label_name"]}')
ax.axis("off")

# 13 bands
for i in range(13):
    ax = plt.subplot(3, 5, i + 2)
    ax.imshow(image_13[i].cpu().numpy(), cmap="gray")
    ax.set_title(SENTINEL2_BANDS[i])
    ax.axis("off")

plt.tight_layout()
plt.show()

## Inspect 1 batch from DataLoader
The goal is to ensure the DataLoader batch is in the correct format to proceed to the CLIP baseline.

In [ ]:
batch = next(iter(loaders["train"]))

print("Batch keys:", batch.keys())
print("Batch image shape:", batch["image"].shape)   # [N, 13, H, W]
print("Batch labels shape:", batch["label"].shape)
print("First 5 label ids:", batch["label"][:5].tolist())
print("First 5 label names:", batch["label_name"][:5])
print("First 2 texts:", batch["text"][:2])
print("First 2 paths:", batch["path"][:2])

rgb_batch = extract_rgb_bands(batch["image"])
print("RGB batch shape:", rgb_batch.shape)          # [N, 3, H, W]